In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# Output folder
# ============================================================
outdir = Path("figures/geometric_settings")
outdir.mkdir(parents=True, exist_ok=True)

# ============================================================
# General plotting utilities
# ============================================================
def setup_ax(ax, title):
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, linewidth=0.7, color="black", alpha=0.35)
    ax.axvline(0, linewidth=0.7, color="black", alpha=0.35)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(r"$\operatorname{Re} z$")
    ax.set_ylabel(r"$\operatorname{Im} z$")
    ax.grid(True, alpha=0.25)

def save_single(fig, filename):
    path = outdir / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

def plot_orientation_arrows(ax, z, n_arrows=3):
    """
    Draw small arrows along a complex curve z.
    """
    if len(z) < 10:
        return
    idxs = np.linspace(len(z)//5, 4*len(z)//5, n_arrows).astype(int)
    for idx in idxs:
        if idx + 3 < len(z):
            ax.annotate(
                "",
                xy=(z[idx+3].real, z[idx+3].imag),
                xytext=(z[idx].real, z[idx].imag),
                arrowprops=dict(arrowstyle="->", linewidth=1.0)
            )

# ============================================================
# 1. Piecewise Lyapunov curve without cusps
#    Smooth arcs meeting with nonzero angles.
# ============================================================
def piecewise_lyapunov_without_cusps():
    # Rounded polygon-like curve built from a polar perturbation.
    theta = np.linspace(0, 2*np.pi, 900)
    r = 1.0 + 0.18*np.cos(3*theta) + 0.08*np.sin(5*theta)
    z = r * np.exp(1j * theta)

    # Add a few visible "corner markers" but no cusps.
    corner_angles = np.array([0.35, 1.65, 3.05, 4.45, 5.45])
    corner_r = 1.0 + 0.18*np.cos(3*corner_angles) + 0.08*np.sin(5*corner_angles)
    corners = corner_r * np.exp(1j * corner_angles)

    fig, ax = plt.subplots(figsize=(5, 5))
    setup_ax(ax, "Piecewise Lyapunov curve without cusps")
    ax.plot(z.real, z.imag, linewidth=2.2)
    ax.scatter(corners.real, corners.imag, s=35, zorder=3)
    plot_orientation_arrows(ax, z, n_arrows=4)
    ax.text(0.02, -1.33, r"$\gamma_j\in(0,2\pi)$", fontsize=11)
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)
    return fig

# ============================================================
# 2. Piecewise Lyapunov curve with first-order cusp
#    A closed curve with a cusp at the origin.
# ============================================================
def piecewise_lyapunov_with_cusp():
    # Cardioid-like curve with cusp at z=0.
    t = np.linspace(0, 2*np.pi, 1000)
    r = 1 - np.cos(t)
    z = r * np.exp(1j * t)

    # Rotate and scale for a clearer view.
    z = 0.55 * z * np.exp(-0.5j)

    fig, ax = plt.subplots(figsize=(5, 5))
    setup_ax(ax, "Piecewise Lyapunov curve with cusp")
    ax.plot(z.real, z.imag, linewidth=2.2)
    ax.scatter([0], [0], s=45, zorder=3)
    ax.text(0.05, 0.05, "cusp", fontsize=10)
    plot_orientation_arrows(ax, z, n_arrows=4)
    ax.set_xlim(-0.35, 1.25)
    ax.set_ylim(-0.95, 0.95)
    return fig

# ============================================================
# 3. Logarithmic star without cusps
#    Gamma_k = { e^{i beta_k} r^{1+i delta} : r in R_+ }.
# ============================================================
def logarithmic_star_without_cusps():
    N = 6
    betas = np.linspace(0, 2*np.pi, N, endpoint=False)
    delta = 0.28
    r = np.linspace(0.08, 1.35, 500)

    fig, ax = plt.subplots(figsize=(5, 5))
    setup_ax(ax, "Logarithmic star without cusps")

    for beta in betas:
        theta = beta + delta * np.log(r)
        z = r * np.exp(1j * theta)
        ax.plot(z.real, z.imag, linewidth=2.0)
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=35, zorder=3)
    ax.text(0.07, 0.06, r"$0$", fontsize=11)
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)
    return fig

# ============================================================
# 4. Pure cusp star-like curve
#    Gamma_k = { (x - i c_k)^(-1) : x in R_+ }.
# ============================================================
def pure_cusp_star():
    c_values = np.array([-3.0, -1.6, -0.8, 0.8, 1.6, 3.0])
    x = np.linspace(0.001, 30.0, 900)

    fig, ax = plt.subplots(figsize=(5, 5))
    setup_ax(ax, "Pure cusp star-like curve")

    for c in c_values:
        z = 1 / (x - 1j*c)
        # Orient toward the common endpoint 0, so x increases.
        ax.plot(z.real, z.imag, linewidth=2.0)
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=45, zorder=4)
    ax.text(0.015, 0.02, r"common point $0$", fontsize=10)
    ax.set_xlim(-0.05, 1.15)
    ax.set_ylim(-1.0, 1.0)
    return fig

# ============================================================
# 5. Mixed star-like curve with cusps and rays
#    Cusp branches: (x - i c_k)^(-1)
#    Ray branches:  e^{i beta_k}/x
# ============================================================
def mixed_star_cusps_and_rays():
    c_values = np.array([-2.4, -1.2, 1.2, 2.4])
    beta_values = np.deg2rad([115, 145, 215, 245])

    x_cusp = np.linspace(0.001, 30.0, 900)
    x_ray = np.linspace(0.35, 30.0, 900)

    fig, ax = plt.subplots(figsize=(5, 5))
    setup_ax(ax, "Mixed star-like curve with cusps and rays")

    # Cusp arcs in the right half-plane.
    for c in c_values:
        z = 1 / (x_cusp - 1j*c)
        ax.plot(z.real, z.imag, linewidth=2.0)
        plot_orientation_arrows(ax, z, n_arrows=1)

    # Rays in the left half-plane.
    for beta in beta_values:
        z = np.exp(1j * beta) / x_ray
        ax.plot(z.real, z.imag, linewidth=2.0, linestyle="--")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=45, zorder=4)
    ax.text(0.03, 0.03, r"$0$", fontsize=11)
    ax.text(0.52, 0.75, "cusps", fontsize=10)
    ax.text(-1.05, 0.75, "rays", fontsize=10)
    ax.set_xlim(-1.35, 1.15)
    ax.set_ylim(-1.15, 1.15)
    return fig

# ============================================================
# Generate individual images
# ============================================================
fig1 = piecewise_lyapunov_without_cusps()
save_single(fig1, "stage1_piecewise_lyapunov_without_cusps.png")

fig2 = piecewise_lyapunov_with_cusp()
save_single(fig2, "stage2_piecewise_lyapunov_with_cusp.png")

fig3 = logarithmic_star_without_cusps()
save_single(fig3, "stage3_logarithmic_star_without_cusps.png")

fig4 = pure_cusp_star()
save_single(fig4, "stage4_pure_cusp_star.png")

fig5 = mixed_star_cusps_and_rays()
save_single(fig5, "stage5_mixed_star_cusps_and_rays.png")

# ============================================================
# Combined panel figure
# ============================================================
fig, axs = plt.subplots(2, 3, figsize=(14, 9))
axs = axs.ravel()

constructors = [
    ("Piecewise Lyapunov\nwithout cusps", piecewise_lyapunov_without_cusps),
    ("Piecewise Lyapunov\nwith cusp", piecewise_lyapunov_with_cusp),
    ("Logarithmic star\nwithout cusps", logarithmic_star_without_cusps),
    ("Pure cusp\nstar-like curve", pure_cusp_star),
    ("Mixed star-like curve\nwith cusps and rays", mixed_star_cusps_and_rays),
]

# Instead of reusing the previous figures, redraw directly into axes.
for ax in axs:
    ax.axis("off")

# Helper functions that draw on a provided axis
def draw_stage1(ax):
    theta = np.linspace(0, 2*np.pi, 900)
    r = 1.0 + 0.18*np.cos(3*theta) + 0.08*np.sin(5*theta)
    z = r * np.exp(1j * theta)
    corner_angles = np.array([0.35, 1.65, 3.05, 4.45, 5.45])
    corner_r = 1.0 + 0.18*np.cos(3*corner_angles) + 0.08*np.sin(5*corner_angles)
    corners = corner_r * np.exp(1j * corner_angles)
    setup_ax(ax, "Piecewise Lyapunov without cusps")
    ax.plot(z.real, z.imag, linewidth=2)
    ax.scatter(corners.real, corners.imag, s=25)
    plot_orientation_arrows(ax, z, n_arrows=3)
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

def draw_stage2(ax):
    t = np.linspace(0, 2*np.pi, 1000)
    r = 1 - np.cos(t)
    z = 0.55 * r * np.exp(1j * t) * np.exp(-0.5j)
    setup_ax(ax, "Piecewise Lyapunov with cusp")
    ax.plot(z.real, z.imag, linewidth=2)
    ax.scatter([0], [0], s=30)
    ax.text(0.05, 0.05, "cusp", fontsize=9)
    plot_orientation_arrows(ax, z, n_arrows=3)
    ax.set_xlim(-0.35, 1.25)
    ax.set_ylim(-0.95, 0.95)

def draw_stage3(ax):
    N = 6
    betas = np.linspace(0, 2*np.pi, N, endpoint=False)
    delta = 0.28
    r = np.linspace(0.08, 1.35, 500)
    setup_ax(ax, "Logarithmic star without cusps")
    for beta in betas:
        theta = beta + delta * np.log(r)
        z = r * np.exp(1j * theta)
        ax.plot(z.real, z.imag, linewidth=1.8)
        plot_orientation_arrows(ax, z, n_arrows=1)
    ax.scatter([0], [0], s=25)
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

def draw_stage4(ax):
    c_values = np.array([-3.0, -1.6, -0.8, 0.8, 1.6, 3.0])
    x = np.linspace(0.001, 30.0, 900)
    setup_ax(ax, "Pure cusp star-like curve")
    for c in c_values:
        z = 1 / (x - 1j*c)
        ax.plot(z.real, z.imag, linewidth=1.8)
        plot_orientation_arrows(ax, z, n_arrows=1)
    ax.scatter([0], [0], s=30)
    ax.set_xlim(-0.05, 1.15)
    ax.set_ylim(-1.0, 1.0)

def draw_stage5(ax):
    c_values = np.array([-2.4, -1.2, 1.2, 2.4])
    beta_values = np.deg2rad([115, 145, 215, 245])
    x_cusp = np.linspace(0.001, 30.0, 900)
    x_ray = np.linspace(0.35, 30.0, 900)
    setup_ax(ax, "Mixed star-like curve")
    for c in c_values:
        z = 1 / (x_cusp - 1j*c)
        ax.plot(z.real, z.imag, linewidth=1.8)
        plot_orientation_arrows(ax, z, n_arrows=1)
    for beta in beta_values:
        z = np.exp(1j * beta) / x_ray
        ax.plot(z.real, z.imag, linewidth=1.8, linestyle="--")
        plot_orientation_arrows(ax, z, n_arrows=1)
    ax.scatter([0], [0], s=30)
    ax.text(0.42, 0.78, "cusps", fontsize=9)
    ax.text(-1.05, 0.78, "rays", fontsize=9)
    ax.set_xlim(-1.35, 1.15)
    ax.set_ylim(-1.15, 1.15)

draws = [draw_stage1, draw_stage2, draw_stage3, draw_stage4, draw_stage5]
for ax, draw in zip(axs[:5], draws):
    draw(ax)

axs[5].axis("off")
axs[5].text(
    0.5, 0.5,
    "Geometric settings\nused in the literature",
    ha="center",
    va="center",
    fontsize=15
)

fig.tight_layout()
combined_path = outdir / "geometric_settings_panel.png"
fig.savefig(combined_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {combined_path}")

Saved: figures/geometric_settings/stage1_piecewise_lyapunov_without_cusps.png
Saved: figures/geometric_settings/stage2_piecewise_lyapunov_with_cusp.png
Saved: figures/geometric_settings/stage3_logarithmic_star_without_cusps.png
Saved: figures/geometric_settings/stage4_pure_cusp_star.png
Saved: figures/geometric_settings/stage5_mixed_star_cusps_and_rays.png
Saved: figures/geometric_settings/geometric_settings_panel.png


In [2]:
def duduchava_1995_curve_with_cusps():
    """
    Figure inspired by Duduchava--Latsabidze--Saginashvili (1995), Fig. 1:
    a closed oriented piecewise-smooth Liapounov curve with knots, angles,
    an outward cusp gamma_1 = 0, and an inward cusp gamma_k = 2*pi.
    """

    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    setup_ax(ax, "Piecewise Lyapunov curve with cusps")

    # ------------------------------------------------------------
    # Main boundary points, chosen to imitate Fig. 1 schematically
    # ------------------------------------------------------------
    c1 = np.array([-2.6, -0.15])   # outward cusp, gamma_1 = 0
    c2 = np.array([-0.45,  1.45])
    cj = np.array([ 2.25,  1.25])
    ck = np.array([ 1.25,  0.10])  # inward cusp, gamma_k = 2 pi
    cn = np.array([-0.45, -1.25])

    # ------------------------------------------------------------
    # Helper: quadratic Bezier curve
    # ------------------------------------------------------------
    def bezier(p0, p1, p2, n=200):
        t = np.linspace(0, 1, n)
        z = ((1 - t)**2)[:, None] * p0 + (2 * (1 - t) * t)[:, None] * p1 + (t**2)[:, None] * p2
        return z[:, 0] + 1j * z[:, 1]

    # Upper arc from c1 to c2
    z12 = bezier(c1, np.array([-1.55, 0.25]), c2, 220)

    # Top arc from c2 to cj
    z2j = bezier(c2, np.array([0.75, 0.95]), cj, 220)

    # Arc from cj to ck
    zjk = bezier(cj, np.array([2.55, 0.55]), ck, 180)

    # Small upper branch into inward cusp ck
    zck_upper = bezier(np.array([2.05, 0.10]), np.array([1.70, 0.15]), ck, 120)

    # Small lower branch leaving inward cusp ck
    zck_lower = bezier(ck, np.array([1.60, -0.35]), np.array([1.05, -0.70]), 120)

    # Lower arc from the cusp region to cn
    zn = bezier(np.array([1.05, -0.70]), np.array([0.35, -1.05]), cn, 220)

    # Lower arc from cn back to c1
    zn1 = bezier(cn, np.array([-1.35, -0.55]), c1, 220)

    # Concatenate main curve
    z = np.concatenate([z12, z2j, zjk, zck_lower, zn, zn1])

    ax.plot(z.real, z.imag, linewidth=2.1, color="black")

    # Draw the upper small branch that ends at ck, to show the inward cusp more clearly
    ax.plot(zck_upper.real, zck_upper.imag, linewidth=2.1, color="black")

    # ------------------------------------------------------------
    # Orientation arrows
    # ------------------------------------------------------------
    def add_arrow_on_curve(curve, frac):
        idx = int(frac * (len(curve) - 8))
        ax.annotate(
            "",
            xy=(curve[idx + 7].real, curve[idx + 7].imag),
            xytext=(curve[idx].real, curve[idx].imag),
            arrowprops=dict(arrowstyle="->", linewidth=1.2, color="black")
        )

    add_arrow_on_curve(z12, 0.55)
    add_arrow_on_curve(zn1, 0.45)

    # ------------------------------------------------------------
    # Mark knots
    # ------------------------------------------------------------
    knots = {
        r"$c_1$": c1,
        r"$c_2$": c2,
        r"$c_j$": cj,
        r"$c_k$": ck,
        r"$c_n$": cn,
    }

    for label, p in knots.items():
        ax.scatter([p[0]], [p[1]], s=22, color="black", zorder=4)
        dx, dy = 0.05, 0.08
        if label == r"$c_1$":
            dx, dy = -0.12, -0.25
        elif label == r"$c_k$":
            dx, dy = 0.08, -0.15
        elif label == r"$c_n$":
            dx, dy = -0.18, -0.25
        ax.text(p[0] + dx, p[1] + dy, label, fontsize=11)

    # ------------------------------------------------------------
    # Interior and exterior labels
    # ------------------------------------------------------------
    ax.text(0.0, 0.18, r"$D^{+}$", fontsize=13)
    ax.text(0.9, 1.45, r"$D^{-}$", fontsize=13)
    ax.text(-1.35, 0.25, r"$\Gamma$", fontsize=12)

    # Angle/cusp labels
    ax.text(-2.0, -0.16, r"$\gamma_1=0$", fontsize=11)
    ax.text(-0.55, 1.05, r"$\gamma_2$", fontsize=11)
    ax.text(1.95, 0.98, r"$\gamma_j$", fontsize=11)
    ax.text(0.92, 0.32, r"$\gamma_k=2\pi$", fontsize=11)
    ax.text(-0.45, -0.78, r"$\gamma_n$", fontsize=11)

    # Origin marker, as in the paper's figure
    ax.scatter([0], [-0.25], s=18, color="black")
    ax.text(-0.08, -0.12, r"$0$", fontsize=11)

    ax.set_xlim(-3.0, 2.8)
    ax.set_ylim(-1.65, 1.8)
    ax.set_xlabel(r"$\operatorname{Re} z$")
    ax.set_ylabel(r"$\operatorname{Im} z$")
    ax.grid(True, alpha=0.18)

    return fig

In [3]:
fig2 = duduchava_1995_curve_with_cusps()
save_single(fig2, "stage2_duduchava_1995_curve_with_cusps.png")

Saved: figures/geometric_settings/stage2_duduchava_1995_curve_with_cusps.png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# Output folder
# ============================================================
outdir = Path("figures/geometric_settings")
outdir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Global figure style: same canvas for every image
# ============================================================
FIGSIZE_SINGLE = (5.2, 5.2)
DPI = 300


# ============================================================
# General plotting utilities
# ============================================================
def new_standard_figure():
    """
    Create a figure with the same canvas size and margins for all images.
    This avoids different visual sizes after exporting.
    """
    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.14, top=0.88)
    return fig, ax


def setup_ax(ax, title):
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, linewidth=0.7, color="black", alpha=0.35)
    ax.axvline(0, linewidth=0.7, color="black", alpha=0.35)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$\operatorname{Re} z$", fontsize=9)
    ax.set_ylabel(r"$\operatorname{Im} z$", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.22)


def save_single(fig, filename):
    """
    Save without bbox_inches='tight' so all images keep the same canvas size.
    """
    path = outdir / filename
    fig.savefig(path, dpi=DPI)
    plt.close(fig)
    print(f"Saved: {path}")


def plot_orientation_arrows(ax, z, n_arrows=3, color="black"):
    """
    Draw small orientation arrows along a complex curve z.
    """
    if len(z) < 10:
        return

    idxs = np.linspace(len(z) // 5, 4 * len(z) // 5, n_arrows).astype(int)

    for idx in idxs:
        if idx + 3 < len(z):
            ax.annotate(
                "",
                xy=(z[idx + 3].real, z[idx + 3].imag),
                xytext=(z[idx].real, z[idx].imag),
                arrowprops=dict(
                    arrowstyle="->",
                    linewidth=1.0,
                    color=color
                )
            )


def bezier(p0, p1, p2, n=200):
    """
    Quadratic Bezier curve through control points p0, p1, p2.
    The points are real 2D vectors and the output is a complex curve.
    """
    t = np.linspace(0, 1, n)
    z = (
        ((1 - t) ** 2)[:, None] * p0
        + (2 * (1 - t) * t)[:, None] * p1
        + (t ** 2)[:, None] * p2
    )
    return z[:, 0] + 1j * z[:, 1]


# ============================================================
# 1. Piecewise Lyapunov curve without cusps
# ============================================================
def piecewise_lyapunov_without_cusps():
    """
    Piecewise Lyapunov curve without cusps.
    This represents the Costabel-type setting: angles appear, but no cusps.
    """

    theta = np.linspace(0, 2 * np.pi, 900)
    r = 1.0 + 0.18 * np.cos(3 * theta) + 0.08 * np.sin(5 * theta)
    z = r * np.exp(1j * theta)

    corner_angles = np.array([0.35, 1.65, 3.05, 4.45, 5.45])
    corner_r = (
        1.0
        + 0.18 * np.cos(3 * corner_angles)
        + 0.08 * np.sin(5 * corner_angles)
    )
    corners = corner_r * np.exp(1j * corner_angles)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Piecewise Lyapunov curve without cusps")

    ax.plot(z.real, z.imag, linewidth=2.2, color="black")
    ax.scatter(corners.real, corners.imag, s=32, color="black", zorder=3)

    plot_orientation_arrows(ax, z, n_arrows=4)

    ax.text(0.03, -1.32, r"$\gamma_j\in(0,2\pi)$", fontsize=10)

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

    return fig


# ============================================================
# 2. Duduchava--Latsabidze--Saginashvili 1995 curve with cusps
# ============================================================
def duduchava_1995_curve_with_cusps():
    """
    Figure inspired by Duduchava--Latsabidze--Saginashvili (1995), Fig. 1:
    a closed oriented piecewise-smooth Liapounov curve with knots, angles,
    an outward cusp gamma_1 = 0, and an inward cusp gamma_k = 2*pi.
    """

    fig, ax = new_standard_figure()
    setup_ax(ax, "Piecewise Lyapunov curve with cusps")

    # Main boundary points, chosen to imitate Fig. 1 schematically.
    c1 = np.array([-2.6, -0.15])   # outward cusp, gamma_1 = 0
    c2 = np.array([-0.45, 1.45])
    cj = np.array([2.25, 1.25])
    ck = np.array([1.25, 0.10])    # inward cusp, gamma_k = 2*pi
    cn = np.array([-0.45, -1.25])

    # Upper arc from c1 to c2.
    z12 = bezier(c1, np.array([-1.55, 0.25]), c2, 220)

    # Top arc from c2 to cj.
    z2j = bezier(c2, np.array([0.75, 0.95]), cj, 220)

    # Arc from cj to the inward cusp ck.
    zjk = bezier(cj, np.array([2.55, 0.55]), ck, 180)

    # Small upper branch into inward cusp ck.
    zck_upper = bezier(
        np.array([2.05, 0.10]),
        np.array([1.70, 0.15]),
        ck,
        120
    )

    # Small lower branch leaving inward cusp ck.
    zck_lower = bezier(
        ck,
        np.array([1.60, -0.35]),
        np.array([1.05, -0.70]),
        120
    )

    # Lower arc from cusp region to cn.
    zn = bezier(
        np.array([1.05, -0.70]),
        np.array([0.35, -1.05]),
        cn,
        220
    )

    # Lower arc from cn back to c1.
    zn1 = bezier(cn, np.array([-1.35, -0.55]), c1, 220)

    # Main curve.
    z = np.concatenate([z12, z2j, zjk, zck_lower, zn, zn1])

    ax.plot(z.real, z.imag, linewidth=2.1, color="black")
    ax.plot(zck_upper.real, zck_upper.imag, linewidth=2.1, color="black")

    # Orientation arrows.
    def add_arrow_on_curve(curve, frac):
        idx = int(frac * (len(curve) - 8))
        ax.annotate(
            "",
            xy=(curve[idx + 7].real, curve[idx + 7].imag),
            xytext=(curve[idx].real, curve[idx].imag),
            arrowprops=dict(
                arrowstyle="->",
                linewidth=1.2,
                color="black"
            )
        )

    add_arrow_on_curve(z12, 0.55)
    add_arrow_on_curve(zn1, 0.45)

    # Mark knots.
    knots = {
        r"$c_1$": c1,
        r"$c_2$": c2,
        r"$c_j$": cj,
        r"$c_k$": ck,
        r"$c_n$": cn,
    }

    for label, p in knots.items():
        ax.scatter([p[0]], [p[1]], s=22, color="black", zorder=4)

        dx, dy = 0.05, 0.08
        if label == r"$c_1$":
            dx, dy = -0.12, -0.25
        elif label == r"$c_k$":
            dx, dy = 0.08, -0.15
        elif label == r"$c_n$":
            dx, dy = -0.18, -0.25

        ax.text(p[0] + dx, p[1] + dy, label, fontsize=10)

    # Interior and exterior labels.
    ax.text(0.0, 0.18, r"$D^{+}$", fontsize=12)
    ax.text(0.9, 1.45, r"$D^{-}$", fontsize=12)
    ax.text(-1.35, 0.25, r"$\Gamma$", fontsize=11)

    # Angle/cusp labels.
    ax.text(-2.0, -0.16, r"$\gamma_1=0$", fontsize=10)
    ax.text(-0.55, 1.05, r"$\gamma_2$", fontsize=10)
    ax.text(1.95, 0.98, r"$\gamma_j$", fontsize=10)
    ax.text(0.92, 0.32, r"$\gamma_k=2\pi$", fontsize=10)
    ax.text(-0.45, -0.78, r"$\gamma_n$", fontsize=10)

    # Origin marker, as in the paper's figure.
    ax.scatter([0], [-0.25], s=18, color="black")
    ax.text(-0.08, -0.12, r"$0$", fontsize=10)

    # Wider x-range but same canvas as the other images.
    ax.set_xlim(-3.15, 2.95)
    ax.set_ylim(-2.15, 2.05)

    return fig


# ============================================================
# 3. Logarithmic star without cusps
# ============================================================
def logarithmic_star_without_cusps():
    """
    Logarithmic star without cusps:
        Gamma_k = { e^{i beta_k} r^{1+i delta} : r in R_+ }.
    """

    N = 6
    betas = np.linspace(0, 2 * np.pi, N, endpoint=False)
    delta = 0.28
    r = np.linspace(0.08, 1.35, 500)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Logarithmic star without cusps")

    for beta in betas:
        theta = beta + delta * np.log(r)
        z = r * np.exp(1j * theta)
        ax.plot(z.real, z.imag, linewidth=2.0, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=35, color="black", zorder=3)
    ax.text(0.07, 0.06, r"$0$", fontsize=10)

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

    return fig


# ============================================================
# 4. Pure cusp star-like curve
# ============================================================
def pure_cusp_star():
    """
    Pure cusp star-like curve:
        Gamma_k = { (x - i c_k)^(-1) : x in R_+ }.
    Each branch is an arc of a circle in the right half-plane.
    """

    c_values = np.array([-3.0, -1.6, -0.8, 0.8, 1.6, 3.0])
    x = np.linspace(0.001, 30.0, 900)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Pure cusp star-like curve")

    for c in c_values:
        z = 1 / (x - 1j * c)
        ax.plot(z.real, z.imag, linewidth=2.0, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=45, color="black", zorder=4)
    ax.text(0.015, 0.02, r"common point $0$", fontsize=9)

    ax.set_xlim(-0.05, 1.15)
    ax.set_ylim(-1.0, 1.0)

    return fig


# ============================================================
# 5. Mixed star-like curve with cusps and rays
# ============================================================
def mixed_star_cusps_and_rays():
    """
    Mixed star-like curve with cusps and rays:
        Cusp branches: Gamma_k = { (x - i c_k)^(-1) : x in R_+ }
        Ray branches:  Gamma_k = { e^{i beta_k}/x : x in R_+ }.
    """

    c_values = np.array([-2.4, -1.2, 1.2, 2.4])
    beta_values = np.deg2rad([115, 145, 215, 245])

    x_cusp = np.linspace(0.001, 30.0, 900)
    x_ray = np.linspace(0.35, 30.0, 900)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Mixed star-like curve")

    # Cusp arcs in the right half-plane.
    for c in c_values:
        z = 1 / (x_cusp - 1j * c)
        ax.plot(z.real, z.imag, linewidth=2.0, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    # Rays in the left half-plane.
    for beta in beta_values:
        z = np.exp(1j * beta) / x_ray
        ax.plot(z.real, z.imag, linewidth=2.0, linestyle="--", color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=45, color="black", zorder=4)
    ax.text(0.03, 0.03, r"$0$", fontsize=10)

    ax.text(0.52, 0.75, "cusps", fontsize=9)
    ax.text(-1.05, 0.75, "rays", fontsize=9)

    ax.set_xlim(-1.35, 1.15)
    ax.set_ylim(-1.15, 1.15)

    return fig


# ============================================================
# Generate individual images
# ============================================================
fig1 = piecewise_lyapunov_without_cusps()
save_single(fig1, "stage1_piecewise_lyapunov_without_cusps.png")

fig2 = duduchava_1995_curve_with_cusps()
save_single(fig2, "stage2_duduchava_1995_curve_with_cusps.png")

fig3 = logarithmic_star_without_cusps()
save_single(fig3, "stage3_logarithmic_star_without_cusps.png")

fig4 = pure_cusp_star()
save_single(fig4, "stage4_pure_cusp_star.png")

fig5 = mixed_star_cusps_and_rays()
save_single(fig5, "stage5_mixed_star_cusps_and_rays.png")


# ============================================================
# Optional: combined panel figure
# ============================================================
def draw_stage1(ax):
    theta = np.linspace(0, 2 * np.pi, 900)
    r = 1.0 + 0.18 * np.cos(3 * theta) + 0.08 * np.sin(5 * theta)
    z = r * np.exp(1j * theta)

    corner_angles = np.array([0.35, 1.65, 3.05, 4.45, 5.45])
    corner_r = (
        1.0
        + 0.18 * np.cos(3 * corner_angles)
        + 0.08 * np.sin(5 * corner_angles)
    )
    corners = corner_r * np.exp(1j * corner_angles)

    setup_ax(ax, "Piecewise Lyapunov without cusps")
    ax.plot(z.real, z.imag, linewidth=2, color="black")
    ax.scatter(corners.real, corners.imag, s=25, color="black")
    plot_orientation_arrows(ax, z, n_arrows=3)

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)


def draw_stage2(ax):
    setup_ax(ax, "Piecewise Lyapunov with cusps")

    c1 = np.array([-2.6, -0.15])
    c2 = np.array([-0.45, 1.45])
    cj = np.array([2.25, 1.25])
    ck = np.array([1.25, 0.10])
    cn = np.array([-0.45, -1.25])

    z12 = bezier(c1, np.array([-1.55, 0.25]), c2, 220)
    z2j = bezier(c2, np.array([0.75, 0.95]), cj, 220)
    zjk = bezier(cj, np.array([2.55, 0.55]), ck, 180)
    zck_upper = bezier(np.array([2.05, 0.10]), np.array([1.70, 0.15]), ck, 120)
    zck_lower = bezier(ck, np.array([1.60, -0.35]), np.array([1.05, -0.70]), 120)
    zn = bezier(np.array([1.05, -0.70]), np.array([0.35, -1.05]), cn, 220)
    zn1 = bezier(cn, np.array([-1.35, -0.55]), c1, 220)

    z = np.concatenate([z12, z2j, zjk, zck_lower, zn, zn1])

    ax.plot(z.real, z.imag, linewidth=2.0, color="black")
    ax.plot(zck_upper.real, zck_upper.imag, linewidth=2.0, color="black")

    for p in [c1, c2, cj, ck, cn]:
        ax.scatter([p[0]], [p[1]], s=18, color="black")

    ax.text(-2.0, -0.16, r"$\gamma_1=0$", fontsize=8)
    ax.text(0.92, 0.32, r"$\gamma_k=2\pi$", fontsize=8)
    ax.text(0.0, 0.18, r"$D^{+}$", fontsize=9)
    ax.text(0.9, 1.45, r"$D^{-}$", fontsize=9)

    ax.set_xlim(-3.15, 2.95)
    ax.set_ylim(-2.15, 2.05)


def draw_stage3(ax):
    N = 6
    betas = np.linspace(0, 2 * np.pi, N, endpoint=False)
    delta = 0.28
    r = np.linspace(0.08, 1.35, 500)

    setup_ax(ax, "Logarithmic star without cusps")

    for beta in betas:
        theta = beta + delta * np.log(r)
        z = r * np.exp(1j * theta)
        ax.plot(z.real, z.imag, linewidth=1.8, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=25, color="black")

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)


def draw_stage4(ax):
    c_values = np.array([-3.0, -1.6, -0.8, 0.8, 1.6, 3.0])
    x = np.linspace(0.001, 30.0, 900)

    setup_ax(ax, "Pure cusp star-like curve")

    for c in c_values:
        z = 1 / (x - 1j * c)
        ax.plot(z.real, z.imag, linewidth=1.8, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=30, color="black")

    ax.set_xlim(-0.05, 1.15)
    ax.set_ylim(-1.0, 1.0)


def draw_stage5(ax):
    c_values = np.array([-2.4, -1.2, 1.2, 2.4])
    beta_values = np.deg2rad([115, 145, 215, 245])

    x_cusp = np.linspace(0.001, 30.0, 900)
    x_ray = np.linspace(0.35, 30.0, 900)

    setup_ax(ax, "Mixed star-like curve")

    for c in c_values:
        z = 1 / (x_cusp - 1j * c)
        ax.plot(z.real, z.imag, linewidth=1.8, color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    for beta in beta_values:
        z = np.exp(1j * beta) / x_ray
        ax.plot(z.real, z.imag, linewidth=1.8, linestyle="--", color="black")
        plot_orientation_arrows(ax, z, n_arrows=1)

    ax.scatter([0], [0], s=30, color="black")
    ax.text(0.42, 0.78, "cusps", fontsize=8)
    ax.text(-1.05, 0.78, "rays", fontsize=8)

    ax.set_xlim(-1.35, 1.15)
    ax.set_ylim(-1.15, 1.15)


# ============================================================
# Combined panel image
# ============================================================
fig, axs = plt.subplots(2, 3, figsize=(14, 9))
axs = axs.ravel()

draw_functions = [
    draw_stage1,
    draw_stage2,
    draw_stage3,
    draw_stage4,
    draw_stage5,
]

for ax, draw in zip(axs[:5], draw_functions):
    draw(ax)

axs[5].axis("off")
axs[5].text(
    0.5,
    0.5,
    "Geometric settings\nused in the literature",
    ha="center",
    va="center",
    fontsize=15
)

fig.tight_layout()
combined_path = outdir / "geometric_settings_panel.png"
fig.savefig(combined_path, dpi=DPI)
plt.close(fig)

print(f"Saved: {combined_path}")

Saved: figures/geometric_settings/stage1_piecewise_lyapunov_without_cusps.png
Saved: figures/geometric_settings/stage2_duduchava_1995_curve_with_cusps.png
Saved: figures/geometric_settings/stage3_logarithmic_star_without_cusps.png
Saved: figures/geometric_settings/stage4_pure_cusp_star.png
Saved: figures/geometric_settings/stage5_mixed_star_cusps_and_rays.png
Saved: figures/geometric_settings/geometric_settings_panel.png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# Output folder
# ============================================================
outdir = Path("figures/geometric_settings")
outdir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Global figure style
# ============================================================
FIGSIZE_SINGLE = (5.2, 5.2)
DPI = 300

COLORS = {
    "curve": "#1f77b4",
    "curve2": "#d62728",
    "curve3": "#2ca02c",
    "curve4": "#9467bd",
    "ray": "#ff7f0e",
    "point": "#111111",
    "axis": "#333333",
    "grid": "#999999",
}


# ============================================================
# General plotting utilities
# ============================================================
def new_standard_figure():
    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.14, top=0.88)
    return fig, ax


def setup_ax(ax, title):
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, linewidth=0.7, color=COLORS["axis"], alpha=0.35)
    ax.axvline(0, linewidth=0.7, color=COLORS["axis"], alpha=0.35)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$\operatorname{Re} z$", fontsize=9)
    ax.set_ylabel(r"$\operatorname{Im} z$", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.22, color=COLORS["grid"])


def save_single(fig, filename):
    path = outdir / filename
    fig.savefig(path, dpi=DPI)
    plt.close(fig)
    print(f"Saved: {path}")


def plot_orientation_arrows(ax, z, n_arrows=3, color=None):
    if color is None:
        color = COLORS["point"]

    if len(z) < 10:
        return

    idxs = np.linspace(len(z) // 5, 4 * len(z) // 5, n_arrows).astype(int)

    for idx in idxs:
        if idx + 3 < len(z):
            ax.annotate(
                "",
                xy=(z[idx + 3].real, z[idx + 3].imag),
                xytext=(z[idx].real, z[idx].imag),
                arrowprops=dict(
                    arrowstyle="->",
                    linewidth=1.0,
                    color=color
                )
            )


def bezier(p0, p1, p2, n=200):
    t = np.linspace(0, 1, n)
    z = (
        ((1 - t) ** 2)[:, None] * p0
        + (2 * (1 - t) * t)[:, None] * p1
        + (t ** 2)[:, None] * p2
    )
    return z[:, 0] + 1j * z[:, 1]


# ============================================================
# 1. Piecewise Lyapunov curve without cusps
# ============================================================
def piecewise_lyapunov_without_cusps():
    theta = np.linspace(0, 2 * np.pi, 900)
    r = 1.0 + 0.18 * np.cos(3 * theta) + 0.08 * np.sin(5 * theta)
    z = r * np.exp(1j * theta)

    corner_angles = np.array([0.35, 1.65, 3.05, 4.45, 5.45])
    corner_r = (
        1.0
        + 0.18 * np.cos(3 * corner_angles)
        + 0.08 * np.sin(5 * corner_angles)
    )
    corners = corner_r * np.exp(1j * corner_angles)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Piecewise Lyapunov curve without cusps")

    ax.plot(z.real, z.imag, linewidth=2.2, color=COLORS["curve"])
    ax.scatter(
        corners.real,
        corners.imag,
        s=38,
        color=COLORS["curve2"],
        edgecolor="white",
        linewidth=0.6,
        zorder=3
    )

    plot_orientation_arrows(ax, z, n_arrows=4, color=COLORS["curve"])
    ax.text(0.03, -1.32, r"$\gamma_j\in(0,2\pi)$", fontsize=10)

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

    return fig


# ============================================================
# 2. Duduchava--Latsabidze--Saginashvili 1995 curve with cusps
# ============================================================
def duduchava_1995_curve_with_cusps():
    fig, ax = new_standard_figure()
    setup_ax(ax, "Piecewise Lyapunov curve with cusps")

    c1 = np.array([-2.6, -0.15])
    c2 = np.array([-0.45, 1.45])
    cj = np.array([2.25, 1.25])
    ck = np.array([1.25, 0.10])
    cn = np.array([-0.45, -1.25])

    z12 = bezier(c1, np.array([-1.55, 0.25]), c2, 220)
    z2j = bezier(c2, np.array([0.75, 0.95]), cj, 220)
    zjk = bezier(cj, np.array([2.55, 0.55]), ck, 180)

    zck_upper = bezier(
        np.array([2.05, 0.10]),
        np.array([1.70, 0.15]),
        ck,
        120
    )

    zck_lower = bezier(
        ck,
        np.array([1.60, -0.35]),
        np.array([1.05, -0.70]),
        120
    )

    zn = bezier(
        np.array([1.05, -0.70]),
        np.array([0.35, -1.05]),
        cn,
        220
    )

    zn1 = bezier(cn, np.array([-1.35, -0.55]), c1, 220)

    z = np.concatenate([z12, z2j, zjk, zck_lower, zn, zn1])

    ax.plot(z.real, z.imag, linewidth=2.2, color=COLORS["curve2"])
    ax.plot(zck_upper.real, zck_upper.imag, linewidth=2.2, color=COLORS["curve2"])

    def add_arrow_on_curve(curve, frac):
        idx = int(frac * (len(curve) - 8))
        ax.annotate(
            "",
            xy=(curve[idx + 7].real, curve[idx + 7].imag),
            xytext=(curve[idx].real, curve[idx].imag),
            arrowprops=dict(
                arrowstyle="->",
                linewidth=1.2,
                color=COLORS["curve2"]
            )
        )

    add_arrow_on_curve(z12, 0.55)
    add_arrow_on_curve(zn1, 0.45)

    knots = {
        r"$c_1$": c1,
        r"$c_2$": c2,
        r"$c_j$": cj,
        r"$c_k$": ck,
        r"$c_n$": cn,
    }

    for label, p in knots.items():
        ax.scatter(
            [p[0]],
            [p[1]],
            s=28,
            color=COLORS["point"],
            edgecolor="white",
            linewidth=0.5,
            zorder=4
        )

        dx, dy = 0.05, 0.08
        if label == r"$c_1$":
            dx, dy = -0.12, -0.25
        elif label == r"$c_k$":
            dx, dy = 0.08, -0.15
        elif label == r"$c_n$":
            dx, dy = -0.18, -0.25

        ax.text(p[0] + dx, p[1] + dy, label, fontsize=10)

    ax.text(0.0, 0.18, r"$D^{+}$", fontsize=12)
    ax.text(0.9, 1.45, r"$D^{-}$", fontsize=12)
    ax.text(-1.35, 0.25, r"$\Gamma$", fontsize=11)

    ax.text(-2.0, -0.16, r"$\gamma_1=0$", fontsize=10, color=COLORS["curve2"])
    ax.text(-0.55, 1.05, r"$\gamma_2$", fontsize=10)
    ax.text(1.95, 0.98, r"$\gamma_j$", fontsize=10)
    ax.text(0.92, 0.32, r"$\gamma_k=2\pi$", fontsize=10, color=COLORS["curve2"])
    ax.text(-0.45, -0.78, r"$\gamma_n$", fontsize=10)

    ax.scatter([0], [-0.25], s=22, color=COLORS["point"])
    ax.text(-0.08, -0.12, r"$0$", fontsize=10)

    ax.set_xlim(-3.15, 2.95)
    ax.set_ylim(-2.15, 2.05)

    return fig


# ============================================================
# 3. Logarithmic star without cusps
# ============================================================
def logarithmic_star_without_cusps():
    N = 6
    betas = np.linspace(0, 2 * np.pi, N, endpoint=False)
    delta = 0.28
    r = np.linspace(0.08, 1.35, 500)

    palette = [
        "#1f77b4",
        "#ff7f0e",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#17becf",
    ]

    fig, ax = new_standard_figure()
    setup_ax(ax, "Logarithmic star without cusps")

    for beta, color in zip(betas, palette):
        theta = beta + delta * np.log(r)
        z = r * np.exp(1j * theta)
        ax.plot(z.real, z.imag, linewidth=2.1, color=color)
        plot_orientation_arrows(ax, z, n_arrows=1, color=color)

    ax.scatter([0], [0], s=40, color=COLORS["point"], zorder=3)
    ax.text(0.07, 0.06, r"$0$", fontsize=10)

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)

    return fig


# ============================================================
# 4. Pure cusp star-like curve
# ============================================================
def pure_cusp_star():
    c_values = np.array([-3.0, -1.6, -0.8, 0.8, 1.6, 3.0])

    palette = [
        "#1f77b4",
        "#17becf",
        "#2ca02c",
        "#ff7f0e",
        "#d62728",
        "#9467bd",
    ]

    x = np.linspace(0.001, 30.0, 900)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Pure cusp star-like curve")

    for c, color in zip(c_values, palette):
        z = 1 / (x - 1j * c)
        ax.plot(z.real, z.imag, linewidth=2.1, color=color)
        plot_orientation_arrows(ax, z, n_arrows=1, color=color)

    ax.scatter([0], [0], s=45, color=COLORS["point"], zorder=4)
    ax.text(0.015, 0.02, r"common point $0$", fontsize=9)

    ax.set_xlim(-0.05, 1.15)
    ax.set_ylim(-1.0, 1.0)

    return fig


# ============================================================
# 5. Mixed star-like curve with cusps and rays
# ============================================================
def mixed_star_cusps_and_rays():
    c_values = np.array([-2.4, -1.2, 1.2, 2.4])
    beta_values = np.deg2rad([115, 145, 215, 245])

    cusp_colors = ["#1f77b4", "#17becf", "#2ca02c", "#9467bd"]
    ray_colors = ["#ff7f0e", "#d62728", "#8c564b", "#e377c2"]

    x_cusp = np.linspace(0.001, 30.0, 900)
    x_ray = np.linspace(0.35, 30.0, 900)

    fig, ax = new_standard_figure()
    setup_ax(ax, "Mixed star-like curve")

    for c, color in zip(c_values, cusp_colors):
        z = 1 / (x_cusp - 1j * c)
        ax.plot(z.real, z.imag, linewidth=2.1, color=color)
        plot_orientation_arrows(ax, z, n_arrows=1, color=color)

    for beta, color in zip(beta_values, ray_colors):
        z = np.exp(1j * beta) / x_ray
        ax.plot(z.real, z.imag, linewidth=2.1, linestyle="--", color=color)
        plot_orientation_arrows(ax, z, n_arrows=1, color=color)

    ax.scatter([0], [0], s=45, color=COLORS["point"], zorder=4)
    ax.text(0.03, 0.03, r"$0$", fontsize=10)

    ax.text(0.52, 0.75, "cusps", fontsize=9, color=COLORS["curve"])
    ax.text(-1.05, 0.75, "rays", fontsize=9, color=COLORS["ray"])

    ax.set_xlim(-1.35, 1.15)
    ax.set_ylim(-1.15, 1.15)

    return fig


# ============================================================
# Generate individual images
# ============================================================
fig1 = piecewise_lyapunov_without_cusps()
save_single(fig1, "stage1_piecewise_lyapunov_without_cusps.png")

fig2 = duduchava_1995_curve_with_cusps()
save_single(fig2, "stage2_duduchava_1995_curve_with_cusps.png")

fig3 = logarithmic_star_without_cusps()
save_single(fig3, "stage3_logarithmic_star_without_cusps.png")

fig4 = pure_cusp_star()
save_single(fig4, "stage4_pure_cusp_star.png")

fig5 = mixed_star_cusps_and_rays()
save_single(fig5, "stage5_mixed_star_cusps_and_rays.png")

Saved: figures/geometric_settings/stage1_piecewise_lyapunov_without_cusps.png
Saved: figures/geometric_settings/stage2_duduchava_1995_curve_with_cusps.png
Saved: figures/geometric_settings/stage3_logarithmic_star_without_cusps.png
Saved: figures/geometric_settings/stage4_pure_cusp_star.png
Saved: figures/geometric_settings/stage5_mixed_star_cusps_and_rays.png
